In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [9]:
X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([353, 10]) torch.Size([353, 1])
torch.Size([89, 10]) torch.Size([89, 1])


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.AGD(
    model=model,
    lr=1,
    radius=1000,
    fletcher=False,
    c1=1e-4,
    tau=0.5,
    line_search_method="backtrack",
    line_search_cond="armijo",
    batch_size=1000
    # batch_size=None
)

all_loss = []
patience = 0
max_patience = 10
for epoch in range(100):
    print("epoch: ", epoch, end="")
    all_loss.append(0)
    for batch_idx, (b_x, b_y) in enumerate(data_loader):
        pre = model(b_x)
        loss = loss_fn(pre, b_y)
        opt.zero_grad()
        loss.backward()

        # parameter update step based on optimizer
        opt.step(b_x, b_y, loss_fn)

        all_loss[epoch] += loss

    all_loss[epoch] /= len(data_loader)

    if epoch > 0 and all_loss[epoch - 1] <= all_loss[epoch]:
        patience -= 1
    else:
        patience = max_patience

    print(", loss: {}".format(all_loss[epoch].detach().cpu().numpy().item()))

    if patience <= 0:
        break

epoch:  0, loss: 0.07857643812894821
epoch:  1, loss: 0.07595063000917435
epoch:  2, loss: 0.06734173744916916
epoch:  3, loss: 0.055566057562828064
epoch:  4, loss: 0.029168665409088135
epoch:  5, loss: 0.02680475264787674
epoch:  6, loss: 0.026616444811224937
epoch:  7, loss: 0.02632446400821209
epoch:  8, loss: 0.025583213195204735
epoch:  9, loss: 0.025555208325386047
epoch:  10, loss: 0.02539193257689476
epoch:  11, loss: 0.025234103202819824
epoch:  12, loss: 0.02481957897543907
epoch:  13, loss: 0.024663865566253662
epoch:  14, loss: 0.02452847547829151
epoch:  15, loss: 0.024264106526970863
epoch:  16, loss: 0.023978495970368385
epoch:  17, loss: 0.023821404203772545
epoch:  18, loss: 0.023643098771572113
epoch:  19, loss: 0.02360931970179081
epoch:  20, loss: 0.02342798002064228
epoch:  21, loss: 0.023133957758545876
epoch:  22, loss: 0.02302924543619156
epoch:  23, loss: 0.02271970361471176
epoch:  24, loss: 0.022477736696600914
epoch:  25, loss: 0.02210649847984314
epoch:  2

In [19]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6638918296740828
Test metrics:  R2 = -0.0028036766283185965


In [20]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.AGD(
    model=model,
    lr=1,
    mu=0.001,
    mu_dec=0.1,
    fletcher=True,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    solver="pinv",
    batch_size=100
)

all_loss = []
patience = 0
max_patience = 10
for epoch in range(100):
    print("epoch: ", epoch, end="")
    all_loss.append(0)
    for batch_idx, (b_x, b_y) in enumerate(data_loader):
        pre = model(b_x)
        loss = loss_fn(pre, b_y)
        opt.zero_grad()
        loss.backward()

        # parameter update step based on optimizer
        opt.step(b_x, b_y, loss_fn)

        all_loss[epoch] += loss

    all_loss[epoch] /= len(data_loader)

    if epoch > 0 and all_loss[epoch - 1] <= all_loss[epoch]:
        patience -= 1
    else:
        patience = max_patience

    print(", loss: {}".format(all_loss[epoch].cpu().detach().numpy().item()))

    if patience <= 0:
        break

epoch:  0, loss: 0.12270516902208328
epoch:  1, loss: 0.11046756058931351
epoch:  2, loss: 0.07234041392803192
epoch:  3, loss: 0.07063378393650055
epoch:  4, loss: 0.06949543207883835
epoch:  5, loss: 0.06620103120803833
epoch:  6, loss: 0.06345489621162415
epoch:  7, loss: 0.04384242743253708
epoch:  8, loss: 0.0373513363301754
epoch:  9, loss: 0.03490320220589638
epoch:  10, loss: 0.033559318631887436
epoch:  11, loss: 0.030214304104447365
epoch:  12, loss: 0.028148764744400978
epoch:  13, loss: 0.026967531070113182
epoch:  14, loss: 0.02613234892487526
epoch:  15, loss: 0.02602018415927887
epoch:  16, loss: 0.025922369211912155
epoch:  17, loss: 0.02493094839155674
epoch:  18, loss: 0.024851450696587563
epoch:  19, loss: 0.024353982880711555
epoch:  20, loss: 0.02409212291240692
epoch:  21, loss: 0.023836849257349968
epoch:  22, loss: 0.023778196424245834
epoch:  23, loss: 0.023542651906609535
epoch:  24, loss: 0.02350580506026745
epoch:  25, loss: 0.023472590371966362
epoch:  26, 

In [21]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.4621867649697242
Test metrics:  R2 = -0.25540361443946424
